# Crushed Keyz LoRA — Google Colab

Fine-tune **Stable Audio 3 `medium-base`** on your Crushed Keyz melody loops.

**Run order:**
1. **Runtime → Change runtime type → T4 GPU**
2. **Cell 1 — Setup** (~5–10 min)
3. **Runtime → Restart session** → **Cell 2** (verify GPU)
4. **Cell 3 — Hugging Face login** (accept licenses first — see cell)
5. Dataset → train

**HF licenses (one-time, in browser):** [medium-base](https://huggingface.co/stabilityai/stable-audio-3-medium-base) · [medium](https://huggingface.co/stabilityai/stable-audio-3-medium)

Dataset: [crushed_keyz_lora.zip](https://drive.google.com/file/d/1ohWvijQt87ncUsbrQDoz2jw1xPDmZVBG/view?usp=sharing)

In [1]:
# CELL 1 — Setup (no torch imports). Re-run only if /content/stable-audio-3 is missing.
import os

PT = 'https://download.pytorch.org/whl/cu126'

if not os.path.isdir('/content/stable-audio-3'):
    !git clone --depth 1 https://github.com/Stability-AI/stable-audio-3.git /content/stable-audio-3
else:
    print('Repo already cloned — skipping git clone')

%cd /content/stable-audio-3
!pip install -q -e ".[lora]"
!pip install -q torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url {PT} --force-reinstall

print('=' * 60)
print('Setup finished.')
print('Next: Runtime → Restart session  (or Restart kernel in Cursor)')
print('Then run Cell 2 — do NOT re-run Cell 1 unless install failed.')
print('=' * 60)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 143.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 136.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 120.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 38.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 55.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 77.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 55.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 50.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 62.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 59.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.3/201.3 MB 69.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━

: 

: 

: 

In [1]:
# CELL 2 — Run after kernel restart from Cell 1
%cd /content/stable-audio-3

import torch, torchvision, pytorch_lightning as pl

print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| pl', pl.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    raise RuntimeError('No GPU — Runtime → Change runtime type → T4 GPU')

/content/stable-audio-3
torch 2.7.1+cu126 | torchvision 0.22.1+cu126 | pl 2.5.5
CUDA: True
GPU: Tesla T4
VRAM (GB): 15.6


In [14]:
# Hugging Face — paste your token below (won't prompt if one is already cached)
# Create token: https://huggingface.co/settings/tokens (Read access)
# Accept licenses: stable-audio-3-medium-base + stable-audio-3-medium

HF_TOKEN = "hf_REPLACE_ME_WITH_YOUR_TOKEN"  # ← replace with your hf_... token

import os
from pathlib import Path
from huggingface_hub import login, logout, HfApi

# Clear any cached token so this cell always uses HF_TOKEN above
logout()
for key in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
    os.environ.pop(key, None)
for path in (
    Path.home() / ".cache/huggingface/token",
    Path.home() / ".huggingface/token",
):
    if path.exists():
        path.unlink()

if HF_TOKEN == "hf_PASTE_YOUR_TOKEN_HERE" or not HF_TOKEN.startswith("hf_"):
    raise ValueError("Set HF_TOKEN to your real token (hf_...) in this cell")

login(token=HF_TOKEN, add_to_git_credential=False)

api = HfApi(token=HF_TOKEN)
for repo in (
    "stabilityai/stable-audio-3-medium-base",
    "stabilityai/stable-audio-3-medium",
):
    api.model_info(repo)
    print(f"OK — access to {repo}")

OK — access to stabilityai/stable-audio-3-medium-base
OK — access to stabilityai/stable-audio-3-medium


In [4]:
# Already installed by Cell 1 — run only if you skipped setup or on a fresh clone path
import os
assert os.path.isdir('/content/stable-audio-3'), 'Run Cell 1 (Setup) first'
%cd /content/stable-audio-3
print('Repo ready:', os.getcwd())

/content/stable-audio-3
Repo ready: /content/stable-audio-3


In [5]:
# Download dataset from public Google Drive link (works in Cursor + Colab plugin)
import zipfile
from pathlib import Path

DATA_DIR = Path('data/crushed_keyz_lora')
ZIP_PATH = Path('crushed_keyz_lora.zip')

# Public link: https://drive.google.com/file/d/1ohWvijQt87ncUsbrQDoz2jw1xPDmZVBG/view?usp=sharing
DRIVE_FILE_ID = '1ohWvijQt87ncUsbrQDoz2jw1xPDmZVBG'
# Fallback path on your Drive if direct download fails
DRIVE_ZIP = '/content/drive/MyDrive/2026/SoundSauce/training/crushed_keyz_lora.zip'

if not DATA_DIR.is_dir():
    !mkdir -p data
    zip_src = ZIP_PATH

    if not zip_src.is_file():
        !pip install -q gdown
        import gdown
        print('Downloading from public Drive link (~210 MB)...')
        gdown.download(
            f'https://drive.google.com/uc?id={DRIVE_FILE_ID}',
            str(ZIP_PATH),
            quiet=False,
            fuzzy=True,
        )

    if not zip_src.is_file():
        print('Direct download failed — mounting Google Drive...')
        from google.colab import drive
        drive.mount('/content/drive')
        zip_src = Path(DRIVE_ZIP)
        if not zip_src.is_file():
            raise FileNotFoundError(
                f'Not found: {DRIVE_ZIP}\n'
                'Check sharing on the zip, or re-run after fixing Drive path.'
            )

    with zipfile.ZipFile(zip_src, 'r') as zf:
        zf.extractall('data')

mp3s = list(DATA_DIR.glob('*.mp3'))
txts = [p for p in DATA_DIR.glob('*.txt') if p.name != 'MANIFEST.txt']
print(f'Clips: {len(mp3s)} audio, {len(txts)} captions')
assert len(mp3s) == len(txts), 'Each audio file needs a matching .txt caption'

Clips: 46 audio, 46 captions


In [ ]:
# Optional: browser upload (only if notebook runs at colab.research.google.com in Chrome)
# Skip this cell when using Cursor — the upload widget does not work there.
# from google.colab import files
# uploaded = files.upload()  # pick crushed_keyz_lora.zip
# !mkdir -p data && unzip -q "$(ls *.zip | head -1)" -d data

In [ ]:
# (removed — use Cell 1 setup + kernel restart instead of pip+import in one cell)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 116.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 202.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 141.9 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 56.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 60.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 75.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 145.9 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 84.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 59.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 95.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.3/201.3 MB 92.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━

AttributeError: partially initialized module 'torchvision' has no attribute 'extension' (most likely due to a circular import)

In [15]:
# Start training in BACKGROUND — avoids ^C from Cursor/Colab disconnecting long cells
import os
import subprocess
from pathlib import Path

try:
    HF_TOKEN
except NameError:
    raise NameError("Run the HF login cell first (HF_TOKEN must be set)")

STEPS = 500          # smoke test; use 2000 for full run
SAVE_DIR = "lora_out/crushed_keyz"
LOG = Path("lora_train.log")
PID_FILE = Path("lora_train.pid")

os.chdir("/content/stable-audio-3")
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env["HF_TOKEN"] = HF_TOKEN
env["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
env["PYTHONUNBUFFERED"] = "1"

if PID_FILE.exists():
    try:
        os.kill(int(PID_FILE.read_text().strip()), 15)
        print("Stopped previous training PID")
    except (ProcessLookupError, ValueError):
        pass

with open(LOG, "w") as log_f:
    proc = subprocess.Popen(
        [
            "python", "-u", "scripts/train_lora.py",
            "--model", "medium-base",
            "--data_dir", "data/crushed_keyz_lora",
            "--duration", "30",
            "--rank", "16",
            "--adapter_type", "dora-rows",
            "--base_precision", "bf16",
            "--exclude", "seconds_total",
            "--steps", str(STEPS),
            "--batch_size", "1",
            "--checkpoint_every", "250",
            "--demo_every", "250",
            "--save_dir", SAVE_DIR,
            "--name", "crushed-keyz-lora",
        ],
        stdout=log_f,
        stderr=subprocess.STDOUT,
        env=env,
        start_new_session=True,
    )

PID_FILE.write_text(str(proc.pid))
print(f"Training started — PID {proc.pid}")
print(f"Log: /content/stable-audio-3/{LOG}")
print("Run the NEXT cell to watch progress + GPU.")
print("Log may stay quiet 10–20 min during model load — check GPU VRAM in watch cell.")

Training started — PID 25362
Log: /content/stable-audio-3/lora_train.log
Run the NEXT cell to watch progress.
Do not click Stop on this notebook while training runs.
(flash_attn warnings are OK; first 5–15 min may look quiet while weights load)


In [21]:
# Watch training + GPU check (re-run every few minutes)
from pathlib import Path
import os
import time

LOG = Path("/content/stable-audio-3/lora_train.log")
PID_FILE = Path("/content/stable-audio-3/lora_train.pid")

if LOG.exists():
    stat = LOG.stat()
    print(f"Log size: {stat.st_size // 1024} KB  (updated {time.ctime(stat.st_mtime)})")
    lines = LOG.read_text(errors="replace").splitlines()
    print("\n".join(lines[-50:]))
else:
    print("No log yet — run the training cell first")

if PID_FILE.exists():
    pid = int(PID_FILE.read_text().strip())
    try:
        os.kill(pid, 0)
        print(f"\nStatus: running (PID {pid})")
    except ProcessLookupError:
        print(f"\nStatus: finished or stopped (was PID {pid})")

print("\n--- GPU (python should use several GB VRAM when training/load is active) ---")
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv 2>/dev/null | head -5

ckpts = sorted(Path("/content/stable-audio-3/lora_out/crushed_keyz").glob("*.ckpt"))
if ckpts:
    print("\nCheckpoints:", ", ".join(p.name for p in ckpts[-3:]))

Seed set to 42
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)

Status: running (PID 25362)


In [ ]:
# List checkpoints
import glob
ckpts = sorted(glob.glob(f'{SAVE_DIR}/*.ckpt'))
for c in ckpts:
    print(c)
LATEST = ckpts[-1] if ckpts else None
LATEST

In [ ]:
# Quick inference test on Colab
PROMPT = 'cKz! Soulful Shadows Cmin 95 @crushed_key, sparse melodic keys loop'

!stable-audio \
  --model medium-base \
  --lora-ckpt-path {LATEST} \
  -p "{PROMPT}" \
  --duration 30 \
  --steps 8 \
  --cfg 1.0 \
  --seed 42 \
  --out ckz_lora_test.wav

from IPython.display import Audio
Audio('ckz_lora_test.wav')

In [ ]:
# Download LoRA checkpoint to your Mac
from google.colab import files
if LATEST:
    files.download(LATEST)
else:
    print('No checkpoint found — run training first')